# Lab 2 — The Agent Loop, Exposed

**TDWI Transform 2026 · Hands-On: Engineering Agentic AI Solutions**

**Estimated time:** 50 minutes  
**Platform:** Google Colab  
**Coding level:** Basic Python familiarity is helpful. You do not need to be an expert.

In Lab 1, you built a reconciliation investigator visually in Langflow. In this lab, you will rebuild the same investigation in plain Python so you can see the engineering controls a visual framework normally hides.

You are not being tested on Python syntax. Most of the code is already written. When you see **YOUR TURN**, copy and paste the provided code, run it, and inspect what changed.

The goal is to think like an agent engineer:

- What should the model decide?
- What should deterministic code decide?
- What state needs to persist between steps?
- What does "done" mean?
- What should happen when the agent cannot finish?
- What controls prevent the system from wandering or repeating itself?

## Scenario Recap

Morrow Peak Outfitters is trying to explain a recurring quarterly reporting problem.

| Report | Q3 U.S. Direct Net Revenue |
|---|---:|
| Finance executive report | $4,200,000 |
| Sales QBR | $3,800,000 |
| Difference | $400,000 |

The Lab 1 investigator found one supported cause:

**Refund recognition timing explains $180,000 of the $400,000 difference.**

That finding is true. The question for this lab is whether the system should consider the investigation finished.

We are using the same Finance and Sales extracts from Lab 1.

## 1. The Harness: The Software Around the Model

The language model is only one part of an agent.

The **agent harness** is the surrounding software that controls how the task is executed. It determines what capabilities the model can reach, what information is carried forward, how model decisions become actions, and when execution should stop.

In this notebook, the harness is intentionally small enough to read end to end:

```text
User request
     ↓
Investigation state
     ↓
Model chooses the next action
     ↓
Tool registry → tool executes
     ↓
State is updated
     ↓
Stopping policy
     ├── continue
     └── stop
```

By the end of the lab, you will have changed both **business control logic** and an **execution safeguard** without changing the underlying model.

## 2. Set Up the Notebook

Run the next cell. It installs the Gemini SDK and the small set of packages used in this lab.

In [ ]:
# Install only the packages this notebook needs.
# Calling pip through the active Python interpreter keeps this cell valid Python
# and makes the setup easier to reuse outside Colab.
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "google-genai",
    "pandas",
    "pydantic",
])

print("Lab dependencies installed.")


### Configure the workshop API key

Use the temporary Gemini API key provided by the instructor. The key is entered securely at runtime and is not written into the notebook.

In [ ]:
from getpass import getpass
from google import genai

# Keep credentials outside the notebook source so the file can be shared safely.
GEMINI_API_KEY = getpass("Enter the workshop Gemini API key: ")

# Use the same workshop model as Lab 1.
MODEL_NAME = "gemini-3.8-flash"

# One client is used for all model calls in this notebook.
client = genai.Client(api_key=GEMINI_API_KEY)

print(f"Gemini client configured with model: {MODEL_NAME}")

## 3. Load the Lab Data

The notebook first tries to load the two extracts from the workshop GitHub repository.

Before the workshop, the instructor will place the final Finance and Sales CSVs in:

`lab-2-agent-loop/data/`

If the repository copy is unavailable, the cell will prompt you to upload the two CSVs manually.

In [ ]:
import io
import pandas as pd
import requests

# Keep repository paths in one place so file locations are easy to change.
REPO_RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "kieDotson/tdwi-agentic-ai-workshop/main/"
    "lab-2-agent-loop/data"
)

FINANCE_FILE = "morrow_peak_finance_q3_reconciliation_extract.csv"
SALES_FILE = "morrow_peak_sales_q3_reconciliation_extract.csv"


def load_csv_from_repo(filename):
    """Load one workshop CSV from the public GitHub repository."""
    url = f"{REPO_RAW_BASE}/{filename}"
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return pd.read_csv(io.BytesIO(response.content))


try:
    finance_df = load_csv_from_repo(FINANCE_FILE)
    sales_df = load_csv_from_repo(SALES_FILE)
    print("Loaded Finance and Sales extracts from GitHub.")

except Exception:
    # Manual upload is a recovery path if GitHub is unavailable.
    from google.colab import files

    print("Repository files were not available.")
    print("Upload the Finance and Sales CSVs when prompted.")
    uploaded = files.upload()

    finance_df = pd.read_csv(io.BytesIO(uploaded[FINANCE_FILE]))
    sales_df = pd.read_csv(io.BytesIO(uploaded[SALES_FILE]))

print(f"Finance rows: {len(finance_df):,}")
print(f"Sales rows:   {len(sales_df):,}")

### Quick check

We will use the `REPORT_SUMMARY` row as the authoritative reported KPI, just as we did in Lab 1. The detailed refund records give the investigator comparable evidence to inspect.

In [ ]:
display(finance_df.head(5))
display(sales_df.head(5))

## 4. Tools: Plain Python Functions With Descriptions

In Langflow, you connected tools to the Agent component.

Here, each tool has two parts:

1. a Python function that retrieves evidence; and
2. a description that tells the model when the tool is useful.

The functions below intentionally return a compact evidence package rather than every row in the file.

In [ ]:
def load_finance_q3_revenue():
    """
    Return Finance's reported Q3 revenue and comparable refund evidence.

    This tool is intentionally read-only. The agent can inspect evidence
    through the function but cannot modify the underlying source.
    """

    # Pull the authoritative KPI from the report summary.
    summary = finance_df.loc[
        finance_df["record_type"] == "REPORT_SUMMARY",
        ["reported_q3_net_revenue_usd"],
    ].iloc[0].to_dict()

    # Return only the refund fields needed for this investigation.
    # Smaller tool outputs reduce irrelevant context and make traces easier to inspect.
    refunds = finance_df.loc[
        finance_df["record_type"] == "REFUND",
        [
            "refund_id",
            "event_date",
            "event_period",
            "original_order_period",
            "included_in_q3_report",
            "reporting_amount_usd",
        ],
    ].to_dict(orient="records")

    return {
        "source": "Finance",
        "reported_q3_net_revenue_usd": int(
            summary["reported_q3_net_revenue_usd"]
        ),
        "refunds": refunds,
    }


def load_sales_q3_revenue():
    """
    Return Sales' reported Q3 revenue and comparable refund evidence.

    The output shape matches the Finance tool so the harness can process both
    sources consistently even though the business views differ.
    """

    summary = sales_df.loc[
        sales_df["record_type"] == "REPORT_SUMMARY",
        ["reported_q3_net_revenue_usd"],
    ].iloc[0].to_dict()

    refunds = sales_df.loc[
        sales_df["record_type"] == "REFUND",
        [
            "refund_id",
            "event_date",
            "event_period",
            "original_order_period",
            "included_in_q3_report",
            "reporting_amount_usd",
        ],
    ].to_dict(orient="records")

    return {
        "source": "Sales",
        "reported_q3_net_revenue_usd": int(
            summary["reported_q3_net_revenue_usd"]
        ),
        "refunds": refunds,
    }

### The tool registry

The **tool registry** is the list of actions the harness allows the model to request.

The model does not get arbitrary access to Python. It can select from the capabilities we intentionally expose.

In [ ]:
# The registry is the capability boundary for this teaching agent.
# The model may request these actions, but it cannot execute arbitrary Python.
TOOLS = {
    "load_finance_q3_revenue": {
        "description": (
            "Loads the Finance Q3 revenue extract. Use this tool when you need "
            "Finance-reported Q3 revenue, Finance refund treatment, or evidence "
            "from the Finance reporting source."
        ),
        "function": load_finance_q3_revenue,
    },
    "load_sales_q3_revenue": {
        "description": (
            "Loads the Sales Q3 revenue extract. Use this tool when you need "
            "Sales-reported Q3 revenue, Sales refund treatment, or evidence "
            "from the Sales reporting source."
        ),
        "function": load_sales_q3_revenue,
    },
}

for name, tool in TOOLS.items():
    print(f"{name}: {tool['description']}")

## Engineering Checkpoint 1 — Predict Before You Run

Before we build the loop, inspect the tool functions and answer these questions with the person next to you or in your notes:

1. Which values come directly from source data?
2. Which values will need to be calculated by the application?
3. Which decisions actually require model judgment?
4. What information should survive from one model step to the next?

Keep your answers in mind. You will compare them with the state object next.

## 5. State: What the System Knows About the Task

**State** is the structured information the harness keeps while the investigation is running.

Some state comes from the model and tools. Other state is calculated deterministically in Python.

For this reconciliation, we want to track:

- the Finance and Sales reported figures;
- the total discrepancy;
- how much has been explained;
- the remaining residual;
- which sources have been used;
- findings collected so far; and
- how many steps the system has taken.

Keeping these values in structured state means important business facts do not exist only inside the model's prose.

In [ ]:
def new_state():
    """Create a clean state object for one reconciliation run."""

    return {
        # Business facts retrieved from authoritative sources.
        "finance_reported": None,
        "sales_reported": None,

        # Deterministic business state.
        # These values are calculated by Python instead of repeatedly asking
        # the LLM to infer arithmetic from prose.
        "total_gap": None,
        "explained_amount": 0,
        "residual": None,

        # Evidence and execution history carried across agent steps.
        "findings": [],
        "sources_used": [],
        "step_count": 0,
    }


def update_gap(state):
    """
    Recalculate the discrepancy and residual from structured state.

    The model interprets evidence. Python owns the arithmetic invariant:

        residual = total discrepancy - amount explained
    """

    if (
        state["finance_reported"] is not None
        and state["sales_reported"] is not None
    ):
        state["total_gap"] = abs(
            state["finance_reported"]
            - state["sales_reported"]
        )

        state["residual"] = (
            state["total_gap"]
            - state["explained_amount"]
        )

    return state

### Compare Your Prediction

Look back at Engineering Checkpoint 1.

The harness keeps arithmetic such as the total gap and residual in deterministic code. The model will decide what evidence to inspect and how to interpret it.

## 6. The Model's Job: Choose the Next Action

For this lab, we are deliberately keeping the model contract simple and visible.

On each pass, Gemini returns one structured decision:

- load the Finance source;
- load the Sales source; or
- return one evidence-supported finding.

The harness decides whether the requested action is allowed and whether the task should stop.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from google.genai import types


class AgentDecision(BaseModel):
    """The only decision shape the model is allowed to return."""

    action: Literal[
        "load_finance_q3_revenue",
        "load_sales_q3_revenue",
        "final",
    ]

    reason: str = Field(
        description="Short explanation of why this is the next action."
    )

    cause: Optional[str] = Field(
        default=None,
        description="Evidence-supported cause when action is final.",
    )

    amount_usd: Optional[int] = Field(
        default=None,
        description="Amount associated with the supported cause when action is final.",
    )

    explanation: Optional[str] = Field(
        default=None,
        description="Concise business explanation when action is final.",
    )

In [ ]:
import json


SYSTEM_INSTRUCTIONS = """
You are the Reconciliation Investigator for Morrow Peak Outfitters.

Investigate the Q3 U.S. Direct reporting discrepancy using the available
Finance and Sales reporting sources.

Rules:
- Inspect both Finance and Sales before returning a final finding.
- Do not request the same reporting source more than once.
- Use the reported summary values as the official Q3 figures.
- Base every claim on evidence returned by the tools.
- Do not invent missing information.
- Compare refund dates, periods, inclusion behavior, shared identifiers,
  and amounts when useful.
- Once you identify one specific, evidence-supported cause and the amount
  associated with it, you may return a final finding.
"""


def model_decision(state, evidence):
    """
    Ask Gemini for exactly one next action.

    The model receives the instructions, tool descriptions, current state,
    and evidence collected so far. It proposes an action; it does not execute
    application code itself.
    """

    available_tools = {
        name: meta["description"]
        for name, meta in TOOLS.items()
    }

    prompt = f"""
{SYSTEM_INSTRUCTIONS}

Available tools:
{json.dumps(available_tools, indent=2)}

Current investigation state:
{json.dumps(state, indent=2)}

Evidence collected so far:
{json.dumps(evidence, indent=2, default=str)}

Choose the single next action.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            # Structured output gives the application something it can validate.
            response_mime_type="application/json",
            response_schema=AgentDecision,
            # Keep classroom runs reasonably deterministic.
            temperature=0,
        ),
    )

    return AgentDecision.model_validate_json(response.text)

## 7. Tool Dispatch: Turning a Model Decision Into an Action

The model can request a tool by name. The **dispatcher** checks the registry, runs the corresponding Python function, and updates state.

This separation is useful: the model chooses an action, but the application decides what that action actually means.

In [ ]:
def dispatch_tool(tool_name, state, evidence):
    """Execute one allowed tool and update the investigation state."""

    # Never trust a model-produced tool name without checking the registry.
    tool = TOOLS.get(tool_name)

    if tool is None:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    # Execute the application-owned function.
    result = tool["function"]()

    # Save evidence so later model calls can reason over what was already found.
    evidence[tool_name] = result

    # Promote authoritative source values into structured business state.
    if result["source"] == "Finance":
        state["finance_reported"] = result["reported_q3_net_revenue_usd"]
    elif result["source"] == "Sales":
        state["sales_reported"] = result["reported_q3_net_revenue_usd"]

    # Record source usage for both auditability and later safeguards.
    if tool_name not in state["sources_used"]:
        state["sources_used"].append(tool_name)

    # Recalculate deterministic business state whenever new source data arrives.
    update_gap(state)

    return result

## 8. A Readable Trace

Agent systems can become difficult to debug when all you see is the final answer.

For this lab, every step prints the model decision, tool action, and current business state.

In [ ]:
def money(value):
    """Format optional numeric state without breaking when a value is unknown."""
    return "—" if value is None else f"${value:,.0f}"


def print_state(state):
    """Print the business state in a human-readable trace."""

    print("STATE")
    print(f"  Finance reported:  {money(state['finance_reported'])}")
    print(f"  Sales reported:    {money(state['sales_reported'])}")
    print(f"  Total gap:         {money(state['total_gap'])}")
    print(f"  Explained:         {money(state['explained_amount'])}")
    print(f"  Residual:          {money(state['residual'])}")
    print(f"  Sources used:      {state['sources_used']}")
    print()


def sources_exhausted(state):
    """Return True when every currently available business source has been used."""
    return set(state["sources_used"]) == set(TOOLS.keys())

## 9. The Inherited Stopping Policy

The first version mirrors the behavior you saw in Lab 1.

If the model returns a supported finding, the harness accepts that as sufficient reason to stop.

Read the function before you run it.

In [ ]:
def stop_when_model_has_supported_finding(state, decision):
    """
    Stop as soon as the model returns one supported finding.

    This is intentionally the inherited policy for the first run.
    """

    if (
        decision.action == "final"
        and decision.cause
        and decision.amount_usd is not None
    ):
        return True, "finding_returned"

    return False, "continue"

## Engineering Checkpoint 2 — Diagnose the Definition of Done

Before running the agent, answer:

1. What exact condition causes this function to return `True`?
2. Does it inspect `state["residual"]`?
3. Does it distinguish between **found a supported cause** and **finished the reconciliation**?
4. What business condition would you expect a stronger stopping policy to inspect?

Do not change the function yet.

## 10. Run the Agent Loop

This is the entire control loop.

Notice the sequence:

**model decision → tool dispatch → state update → stopping policy**

In [ ]:
def run_agent(stop_policy, tool_guard=None, max_steps=8):
    """
    Run the reconciliation agent with explicit execution controls.

    stop_policy:
        Business rule that decides whether a final finding should terminate the run.

    tool_guard:
        Optional operational rule that can allow or block a requested tool call.

    max_steps:
        Hard execution limit. This protects cost and runtime even if other controls fail.
    """

    state = new_state()
    evidence = {}

    # A hard step cap is an operational safety boundary.
    # It should not be confused with a business definition of success.
    for step in range(1, max_steps + 1):
        state["step_count"] = step

        print("=" * 70)
        print(f"STEP {step}")

        # The model proposes exactly one next action.
        decision = model_decision(state, evidence)

        print(f"Decision: {decision.action}")
        print(f"Reason:   {decision.reason}")
        print()

        # PATH 1: The model requested a tool.
        if decision.action in TOOLS:

            # The harness may veto a model-proposed action before execution.
            if tool_guard is not None:
                allowed, guard_status = tool_guard(
                    decision.action,
                    state,
                )

                if not allowed:
                    print(f"TOOL CALL BLOCKED: {guard_status}")
                    print()

                    # Record the blocked action so the next model step can see
                    # that the request was rejected instead of repeating blindly.
                    evidence.setdefault("_guard_events", []).append({
                        "tool": decision.action,
                        "status": guard_status,
                    })
                    continue

            result = dispatch_tool(
                decision.action,
                state,
                evidence,
            )

            print(f"Tool executed: {decision.action}")
            print(f"Source returned: {result['source']}")
            print_state(state)
            continue

        # PATH 2: The model returned a finding.
        if decision.action == "final":

            # The harness enforces a requirement the model cannot waive:
            # both business sources must be inspected before accepting a finding.
            required_sources = set(TOOLS.keys())
            used_sources = set(state["sources_used"])

            if not required_sources.issubset(used_sources):
                print("Final answer rejected by harness.")
                print("Both Finance and Sales must be inspected first.")
                print()
                continue

            # The model interprets evidence and proposes the amount explained.
            state["explained_amount"] = int(
                decision.amount_usd or 0
            )

            # Python recalculates the residual from structured state.
            update_gap(state)

            state["findings"].append({
                "cause": decision.cause,
                "amount_usd": decision.amount_usd,
                "explanation": decision.explanation,
            })

            # The stopping policy—not the model alone—decides whether this
            # business state is allowed to terminate the run.
            should_stop, status = stop_policy(
                state,
                decision,
            )

            print_state(state)
            print(f"STOP CHECK: {status}")
            print()

            if should_stop:
                return {
                    "state": state,
                    "decision": decision,
                    "status": status,
                    "evidence": evidence,
                }

    # Reaching the hard step limit is an operational terminal state.
    # It does NOT mean the reconciliation succeeded.
    return {
        "state": state,
        "decision": None,
        "status": "max_steps_reached",
        "evidence": evidence,
    }

### Run 1 — inherited behavior

In [ ]:
run_1 = run_agent(
    stop_policy=stop_when_model_has_supported_finding,
    max_steps=8,
)

## 11. Inspect the Result as Structured State

Do not judge only the quality of the prose. Inspect the business state the harness is holding.

In [ ]:
state_1 = run_1["state"]

print("RECONCILIATION STATE")
print("-" * 40)
print(f"Finance reported:      {money(state_1['finance_reported'])}")
print(f"Sales reported:        {money(state_1['sales_reported'])}")
print(f"Total discrepancy:     {money(state_1['total_gap'])}")
print(f"Explained:             {money(state_1['explained_amount'])}")
print(f"Unexplained residual:  {money(state_1['residual'])}")
print(f"Sources exhausted:     {sources_exhausted(state_1)}")
print(f"Stop status:           {run_1['status']}")

## 12. Engineering Checkpoint 3 — Diagnose and Fix Completion

Answer before moving on:

1. How much of the $400,000 discrepancy was explained?
2. How much remains?
3. Were all available sources used?
4. Why did the harness stop anyway?
5. Which part is wrong: the evidence, the model's finding, or the definition of completion?

### YOUR TURN — Replace the Business Stopping Policy

Keep the same model, data, tools, instructions, and model decision contract. Change only the business stopping policy.

### COPY THIS

```python
def business_aware_stop(state, decision):
    if state["residual"] == 0:
        return True, "reconciled"

    if sources_exhausted(state):
        return True, "unresolved"

    return False, "continue"
```

### PASTE IT HERE

Paste the full function into the **next code cell**, directly below the `# PASTE HERE` comment, then run the cell.

The three possible results are:

- `reconciled` — the business objective is complete;
- `unresolved` — the agent should stop because the available evidence is exhausted, but work remains;
- `continue` — useful investigation work is still possible.


In [ ]:
# YOUR TURN
# PASTE HERE



## 13. Run the Same Agent Again

Once you have defined `business_aware_stop`, run the next cell.

In [ ]:
run_2 = run_agent(
    stop_policy=business_aware_stop,
    max_steps=8,
)

In [ ]:
state_2 = run_2["state"]

print("FINAL RECONCILIATION STATUS")
print("-" * 40)
print(f"Finance reported:      {money(state_2['finance_reported'])}")
print(f"Sales reported:        {money(state_2['sales_reported'])}")
print(f"Total discrepancy:     {money(state_2['total_gap'])}")
print(f"Explained:             {money(state_2['explained_amount'])}")
print(f"Unexplained residual:  {money(state_2['residual'])}")
print(f"Sources exhausted:     {sources_exhausted(state_2)}")
print(f"Terminal status:       {run_2['status'].upper()}")

## 14. What Changed?

Compare Run 1 and Run 2.

The model did not change. The tools did not change. The evidence did not change. The supported $180,000 finding did not change.

The harness now assigns a terminal status based on business state rather than treating any supported answer as completion.

An agent can stop correctly without succeeding completely.

## 15. Execution Safety Is a Different Problem

The business stopping policy answers:

**Has the task reached a valid business terminal state?**

An operational safeguard answers:

**Is the agent behaving safely while trying to get there?**

Examples include maximum step counts, duplicate tool-call protection, token budgets, timeouts, and circuit breakers.

You already have a maximum step count. Now you will add a repeated-tool guard.

### YOUR TURN — Add a Repeated-Tool Guard

Now add an operational safeguard. This is separate from the business definition of done.

### COPY THIS

```python
def no_repeat_source_guard(tool_name, state):
    if tool_name in state["sources_used"]:
        return False, "duplicate_source_call"

    return True, "allowed"
```

### PASTE IT HERE

Paste the full function into the **next code cell**, directly below the `# PASTE HERE` comment, then run the cell.

This control does **not** decide whether the reconciliation is complete. It decides whether a proposed action is allowed to execute.


In [ ]:
# YOUR TURN
# PASTE HERE



### Test the Safeguard Before Trusting It

Good controls should be testable without waiting for the model to misbehave.

The next cell marks Finance as already used and then asks the guard whether Finance may be called again.

In [ ]:
# Create a deterministic test case for the repeated-tool guard.
test_state = new_state()
test_state["sources_used"].append(
    "load_finance_q3_revenue"
)

allowed, status = no_repeat_source_guard(
    "load_finance_q3_revenue",
    test_state,
)

print(f"Allowed: {allowed}")
print(f"Status:  {status}")

# A small assertion turns the safeguard into something we can verify automatically.
assert allowed is False
assert status == "duplicate_source_call"

print("Guard test passed.")

### Run 3 — Business Control + Operational Safeguard

Run the same agent with both the business-aware stopping policy and the repeated-tool guard.

In [ ]:
run_3 = run_agent(
    stop_policy=business_aware_stop,
    tool_guard=no_repeat_source_guard,
    max_steps=8,
)

print(f"Terminal status: {run_3['status'].upper()}")

## 16. Engineering Checkpoint 4 — Separate the Controls

For each item below, notice who owns the decision in this lab.

| Decision or control | Owner |
|---|---|
| Choose which source to inspect next | Model |
| Retrieve Finance or Sales evidence | Tool / application |
| Calculate the $400,000 gap | Deterministic application code |
| Interpret refund timing evidence | Model |
| Calculate the residual | Deterministic application code |
| Decide whether the residual is zero | Harness / business control logic |
| Block repeated calls to an unchanged source | Harness / operational safeguard |
| Cap the total number of execution steps | Harness / operational safeguard |
| Produce a concise business explanation | Model |

The model supplies judgment where judgment is useful. The harness owns controls that should be explicit, testable, and enforceable.

## 17. Software Engineering Review

The notebook now uses several practices that matter in real agent systems:

### Clear responsibility boundaries
Tool functions retrieve data. The model proposes actions. The dispatcher executes allowed actions. State carries business facts. The stopping policy governs completion.

### Structured model output
The model returns a defined schema instead of arbitrary text when the application needs to make a control decision.

### Deterministic invariants
The total gap and residual are calculated in Python rather than repeatedly inferred from prose.

### Bounded execution
`max_steps` prevents an unbounded loop even if another control fails.

### Testable safeguards
The repeated-tool guard can be tested directly with a deterministic check.

### Readable traces
The notebook prints decisions, actions, and state transitions so a run can be inspected after the fact.

## 18. Finished Early? Knowledge Expanders

Complete the required lab first. Then choose one experiment.

### A. Lower the maximum step count

```python
run_agent(
    stop_policy=business_aware_stop,
    tool_guard=no_repeat_source_guard,
    max_steps=3,
)
```

What happens if an operational limit is reached before the task reaches a business terminal state?

### B. Remove the repeated-tool guard

Run the agent without `tool_guard`. Does the model repeat a source? If not, why is the guard still useful?

### C. Make the model own the residual

Temporarily remove the deterministic `update_gap()` calculation and ask the model to state the remaining residual. Compare the architecture, not only the answer.

### D. Add a new terminal status

What status would you return if a required tool failed instead of returning evidence? Examples: `tool_failure`, `needs_human_review`, or `blocked`.

## 19. Bridge to Lab 3

The Lab 2 agent now behaves more reliably, but it still cannot explain the full $400,000 discrepancy.

It has:

- a $400,000 total gap;
- $180,000 explained by refund timing;
- a $220,000 residual; and
- no additional sources available.

The next engineering problem is not "make the loop longer."

It is:

**What evidence does this investigation need that the current agent cannot reach?**

In Lab 3, you will expand the investigation surface with additional sources and specialist workers.

## Lab 2 Completion Check

You are ready to move on when you can explain:

- what an agent harness is;
- how a tool registry differs from the model itself;
- what state does during a multi-step task;
- why some business facts should remain deterministic;
- how a stopping condition changes agent behavior;
- why `unresolved` can be a correct terminal state;
- how business completion logic differs from operational safeguards; and
- why maximum-step limits and repeated-tool protection belong in the harness.